In [28]:
import os
import re
import json
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from dotenv import load_dotenv

# SDKs Oficiales
from openai import OpenAI
from anthropic import Anthropic
from google import genai

In [30]:
# 1. Validación de claves de API
openai_key = os.getenv('OPENAI_API_KEY')
anthropic_key = os.getenv('ANTHROPIC_API_KEY')
google_key = os.getenv('GOOGLE_API_KEY')

# 3. Validación al estilo Ed Donner (revisando prefijos)
if openai_key:
    print(f"✅ OpenAI API Key existe y empieza con: {openai_key[:8]}")
else:
    print("❌ OpenAI API Key no configurada")

if anthropic_key:
    print(f"✅ Anthropic API Key existe y empieza con: {anthropic_key[:7]}")
else:
    print("❌ Anthropic API Key no configurada")

if google_key:
    print(f"✅ Google API Key existe y empieza con: {google_key[:4]}")
else:
    print("❌ Google API Key no configurada")

✅ OpenAI API Key existe y empieza con: sk-proj-
✅ Anthropic API Key existe y empieza con: sk-ant-
✅ Google API Key existe y empieza con: AIza


In [16]:
# 2. Inicialización de clientes
client_openai = OpenAI(api_key=openai_key) if openai_key else None
client_anthropic = Anthropic(api_key=anthropic_key) if anthropic_key else None
client_gemini = genai.Client(api_key=google_key) if google_key else None

DB_NAME = "empresa_bi.db"

In [29]:
def init_database():
    """Crea la base de datos de la empresa y puebla tablas con información inicial."""
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        
        # Tabla 1: Inventario de productos
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS inventario (
                id INTEGER PRIMARY KEY,
                producto TEXT UNIQUE,
                categoria TEXT,
                stock INTEGER,
                precio_unitario REAL
            )
        ''')
        
        # Tabla 2: Registro de ventas
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS ventas (
                id INTEGER PRIMARY KEY,
                producto TEXT,
                cantidad INTEGER,
                sucursal TEXT,
                fecha TEXT,
                FOREIGN KEY (producto) REFERENCES inventario(producto)
            )
        ''')
        
        # Insertamos datos de prueba solo si la tabla inventario está vacía
        cursor.execute('SELECT COUNT(*) FROM inventario')
        if cursor.fetchone()[0] == 0:
            productos = [
                ('Laptop Pro 15', 'Cómputo', 12, 22000.0),
                ('Monitor 27 Pulgadas', 'Pantallas', 25, 4500.0),
                ('Teclado Mecánico', 'Accesorios', 50, 1200.0),
                ('Mouse Inalámbrico', 'Accesorios', 80, 600.0),
                ('Silla Ergonómica', 'Mobiliario', 8, 3800.0)
            ]
            cursor.executemany(
                'INSERT INTO inventario (producto, categoria, stock, precio_unitario) VALUES (?, ?, ?, ?)',
                productos
            )
            
            ventas = [
                ('Laptop Pro 15', 2, 'CDMX-Centro', '2026-10-01'),
                ('Monitor 27 Pulgadas', 5, 'Monterrey', '2026-10-02'),
                ('Teclado Mecánico', 10, 'CDMX-Centro', '2026-10-02'),
                ('Laptop Pro 15', 1, 'Guadalajara', '2026-10-03'),
                ('Mouse Inalámbrico', 15, 'Monterrey', '2026-10-04')
            ]
            cursor.executemany(
                'INSERT INTO ventas (producto, cantidad, sucursal, fecha) VALUES (?, ?, ?, ?)',
                ventas
            )
            conn.commit()
            print("📦 Base de datos 'empresa_bi.db' creada y poblada con éxito.")
        else:
            print("ℹ️ La base de datos ya contiene registros.")

# Ejecutamos la función de inicialización
init_database()

ℹ️ La base de datos ya contiene registros.


In [17]:
def validate_sql_query(query: str) -> tuple[bool, str]:
    """
    Valida la consulta SQL para prevenir inyección de código y modificaciones no autorizadas.
    Regresa (True, "") si es segura, o (False, "motivo") si se bloquea.
    """
    clean_query = query.strip().upper()
    
    # 1. Permitir únicamente consultas de lectura (SELECT)
    if not clean_query.startswith("SELECT"):
        return False, "Seguridad: Solo se permiten consultas de lectura (SELECT)."
    
    # 2. Palabras clave prohibidas para evitar modificación o borrado
    forbidden_keywords = [
        "DROP", "DELETE", "INSERT", "UPDATE", "ALTER", 
        "CREATE", "TRUNCATE", "EXEC", "EXECUTE", "--", ";"
    ]
    
    for kw in forbidden_keywords:
        # Buscamos la palabra completa para no bloquear términos válidos
        if re.search(r'\b' + kw + r'\b', clean_query):
            return False, f"Seguridad: Se detectó un comando no permitido en la consulta ({kw})."
            
    return True, ""

In [18]:
def execute_sql_query_safe(query: str) -> tuple[str, pd.DataFrame]:
    """
    Ejecuta una consulta SQL sanitizada y retorna tanto la tabla formateada 
    como el DataFrame de pandas para la gráfica.
    """
    print(f"🛡️ VALIDANDO Y EJECUTANDO SQL: {query}", flush=True)
    
    # Paso de validación de seguridad
    is_safe, error_message = validate_sql_query(query)
    if not is_safe:
        print(f"⚠️ BLOQUEO DE SEGURIDAD: {error_message}")
        return error_message, pd.DataFrame()
        
    try:
        with sqlite3.connect(DB_NAME) as conn:
            df = pd.read_sql_query(query, conn)
            if df.empty:
                return "La consulta no devolvió registros.", pd.DataFrame()
            return df.to_string(index=False), df
    except Exception as e:
        return f"Error de ejecución SQL: {str(e)}", pd.DataFrame()

In [ ]:
sql_tool_schema = {
    "name": "execute_sql_query",
    "description": (
        "Ejecuta una consulta SQL SELECT en la base de datos de BI de la empresa. "
        "Usa esta herramienta cuando necesites obtener cifras de ventas o inventario. "
        "Estructura de la BD: "
        "1. inventario(id, producto, categoria, stock, precio_unitario) "
        "2. ventas(id, producto, cantidad, sucursal, fecha)"
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "La consulta SQL SELECT sanitizada y válida a ejecutar.",
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}



In [ ]:
tools_openai = [{"type": "function", "function": sql_tool_schema}]

In [24]:
def run_agent_openai(prompt: str):
    system_prompt = (
        "Eres un Analista Senior de BI y Ciencia de Datos. "
        "Analiza las preguntas de negocio, consulta la base de datos mediante la herramienta SQL "
        "y proporciona un resumen ejecutivo con hallazgos clave."
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    response = client_openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools_openai
    )
    
    last_df = pd.DataFrame()
    
    # Bucle Agéntico (Tool Calling Loop)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_call = message.tool_calls[0]
        
        if tool_call.function.name == "execute_sql_query":
            args = json.loads(tool_call.function.arguments)
            query_str = args.get("query")
            
            # Ejecutamos con validación de seguridad
            result_text, df_data = execute_sql_query_safe(query_str)
            if not df_data.empty:
                last_df = df_data
                
            messages.append(message)
            messages.append({
                "role": "tool",
                "content": result_text,
                "tool_call_id": tool_call.id
            })
            
            response = client_openai.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=tools_openai
            )
            
    return response.choices[0].message.content, last_df

In [25]:
def generate_chart(df: pd.DataFrame):
    """Genera una gráfica dinámica basada en los datos de la consulta SQL."""
    if df.empty or len(df.columns) < 2:
        return None
    
    fig, ax = plt.subplots(figsize=(8, 4))
    
    # Identificamos columnas numéricas y de texto
    text_cols = df.select_dtypes(include=['object']).columns
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    if len(text_cols) > 0 and len(num_cols) > 0:
        x_col = text_cols[0]
        y_col = num_cols[0]
        ax.bar(df[x_col], df[y_col], color='#1f77b4')
        ax.set_xlabel(x_col.capitalize())
        ax.set_ylabel(y_col.capitalize())
        ax.set_title(f"{y_col.capitalize()} por {x_col.capitalize()}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        return fig
    return None

In [26]:
def main_pipeline(prompt: str, selected_model: str):
    """Punto de entrada para la interfaz de Gradio."""
    if selected_model == "OpenAI (GPT-4o-mini)":
        text_response, df_result = run_agent_openai(prompt)
    else:
        text_response = f"El modelo {selected_model} está configurado en el backend. Ejecutando con motor primario OpenAI..."
        text_response, df_result = run_agent_openai(prompt)
        
    chart_fig = generate_chart(df_result)
    
    return text_response, df_result, chart_fig

In [27]:
# ---- CONSTRUCCIÓN DE LA INTERFAZ CON GRADIO BLOCKS ----
with gr.Blocks(title="Agente Analista BI Senior") as demo:
    gr.Markdown("# 📊 Agente Analista de Datos & BI Corporativo")
    gr.Markdown("Asistente agéntico con protección contra SQL Injection, soporte multi-modelo y visualización dinámica.")
    
    with gr.Row():
        with gr.Column(scale=1):
            input_prompt = gr.Textbox(
                label="Consulta de Negocio:", 
                placeholder="Ej. ¿Cuáles son los 3 productos con mayor stock y qué ventas se han hecho de ellos?",
                lines=4
            )
            model_dropdown = gr.Dropdown(
                choices=["OpenAI (GPT-4o-mini)", "Anthropic (Claude 3.5 Sonnet)", "Google (Gemini 1.5 Flash)"],
                value="OpenAI (GPT-4o-mini)",
                label="Selecciona el Modelo LLM:"
            )
            btn_submit = gr.Button("Analizar Datos 🚀", variant="primary")
            
        with gr.Column(scale=2):
            output_markdown = gr.Markdown(label="Resumen Ejecutivo (Análisis de BI):")
            output_table = gr.Dataframe(label="Datos SQL Extraídos (pandas):")
            output_plot = gr.Plot(label="Visualización Dinámica:")
            
    btn_submit.click(
        fn=main_pipeline,
        inputs=[input_prompt, model_dropdown],
        outputs=[output_markdown, output_table, output_plot]
    )

# Lanzamos la aplicación
demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


🛡️ VALIDANDO Y EJECUTANDO SQL: SELECT producto, SUM(cantidad) AS total_vendido FROM ventas GROUP BY producto ORDER BY total_vendido DESC LIMIT 5;


C:\Users\alber\AppData\Local\Temp\ipykernel_15472\1392122186.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=['object']).columns
